In [ ]:
# Import Required Libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from pyspark.sql.functions import explode, lower, col, collect_set, trim, regexp_replace, count
import pandas as pd
import numpy as np


# In cluster use the HDFS path prefix
# path_prefix = "hdfs:///projects/BDA-12/"
# In local use the local path prefix
path_prefix = "../"

In [ ]:
# Load ingredient and nutrient data
spark = SparkSession.builder.appName('HealthinessScoring').getOrCreate()

# Ingredient schema
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])
df_ing = spark.read.schema(ingredients_schema).parquet(
    f'{path_prefix}/output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet/part-00000-33e03f78-d7e8-41fa-a09f-4a8c4605dc35-c000.snappy.parquet'
)

# Nutrient columns
nutri_cols = [
    'fdc_id', 'energy', 'protein', 'carbs', 'total_fat', 'fiber', 'sugars', 'sodium',
    'cholesterol', 'saturated_fat', 'vitamin_c', 'potassium', 'magnesium',
    'vitamin_a', 'vitamin_d', 'vitamin_e', 'vitamin_k', 'thiamin_b1', 'riboflavin_b2', 'niacin_b3', 'vitamin_b6', 'folate', 'vitamin_b12',
    'calcium', 'iron', 'zinc', 'copper', 'manganese', 'selenium',
    'monounsaturated_fat', 'polyunsaturated_fat', 'trans_fat'
]
df_nutri = spark.read.parquet(f'{path_prefix}/output/nutritional_profiles/part-00000-006c6ee8-befb-47d8-8451-e7dae4de0d2f-c000.snappy.parquet')
df_nutri = df_nutri.select(*nutri_cols)

In [95]:
# Explode and normalize ingredients, then merge with nutrients
pattern = r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]'
df_exploded = df_ing.select('fdc_id', 'description', explode('all_ingredients').alias('ingredient'))
df_exploded = df_exploded.withColumn('ingredient_norm', trim(lower(regexp_replace(col('ingredient'), pattern, ''))))
df_unique = df_exploded.groupBy('fdc_id', 'description').agg(collect_set('ingredient_norm').alias('ingredients_set'))

# Convert to pandas and merge
foods = df_unique.toPandas()
nutri = df_nutri.toPandas()
foods = foods.merge(nutri, on='fdc_id', how='left')

In [ ]:
# Define combined scoring rules
healthy_ingredients = set([
    'whole wheat', 'oat', 'spinach', 'broccoli', 'carrot', 'olive oil', 'almond', 'quinoa', 'lentil', 'chicken breast', 'salmon', 'egg',
    'tomato', 'avocado', 'brown rice', 'beans', 'walnut', 'pumpkin seed', 'chia seed', 'flaxseed', 'greek yogurt', 'blueberry', 'apple', 'pear',
    'cabbage', 'cauliflower', 'zucchini', 'bell pepper', 'garlic', 'onion', 'sweet potato', 'kale', 'arugula', 'mushroom', 'cod', 'tuna', 'sardine',
    'hazelnut', 'cashew', 'pistachio', 'sunflower seed', 'sesame seed', 'turkey', 'beet', 'raspberry', 'strawberry', 'lemon', 'lime', 'orange'
])

unhealthy_ingredients = set([
    'sugar', 'corn syrup', 'palm oil', 'margarine', 'sodium benzoate', 'monosodium glutamate', 'artificial flavor', 'high fructose corn syrup', 'trans fat',
    'hydrogenated oil', 'shortening', 'dextrose', 'fructose', 'glucose syrup', 'aspartame', 'acesulfame k', 'saccharin', 'caramel color', 'red 40', 'yellow 5',
    'blue 1', 'potassium bromate', 'bht', 'bha', 'propyl gallate', 'propylene glycol', 'sorbitol', 'polysorbate 80', 'soy protein isolate', 'refined flour',
    'bleached flour', 'canola oil', 'vegetable oil', 'disodium inosinate', 'disodium guanylate', 'sodium nitrate', 'sodium nitrite', 'tbhq', 'phosphoric acid'
])

def score_ingredient(ingredient):
    if ingredient in healthy_ingredients:
        return 1
    elif ingredient in unhealthy_ingredients:
        return -1
    else:
        return 0

def score_nutrients(row):
    score = 0
    # Reward protein, fiber, vitamin_c, potassium, magnesium
    if pd.notnull(row.get('protein')):
        score += row['protein'] * 0.5
    if pd.notnull(row.get('fiber')):
        score += row['fiber'] * 0.7
    if pd.notnull(row.get('vitamin_c')):
        score += row['vitamin_c'] * 0.05
    if pd.notnull(row.get('potassium')):
        score += row['potassium'] * 0.0005
    if pd.notnull(row.get('magnesium')):
        score += row['magnesium'] * 0.01
    # Penalize sugar, sodium, total_fat, cholesterol, saturated_fat
    if pd.notnull(row.get('sugars')):
        score -= row['sugars'] * 0.4
    if pd.notnull(row.get('sodium')):
        score -= row['sodium'] * 0.002
    if pd.notnull(row.get('total_fat')):
        score -= row['total_fat'] * 0.2
    if pd.notnull(row.get('cholesterol')):
        score -= row['cholesterol'] * 0.01
    if pd.notnull(row.get('saturated_fat')):
        score -= row['saturated_fat'] * 0.2
    return score

# Nutri-Score scoring function
def nutri_score(row):
    score = 0
    # Negative points (unhealthy): energy, sugars, saturated fat, sodium
    if pd.notnull(row.get('energy')):
        score += row['energy'] * 0.003
    if pd.notnull(row.get('sugars')):
        score += row['sugars'] * 0.5
    if pd.notnull(row.get('saturated_fat')):
        score += row['saturated_fat'] * 1.0
    if pd.notnull(row.get('sodium')):
        score += row['sodium'] * 0.004
    # Positive points (healthy): fiber, protein, fruits/vegetables (approx. by vitamin C, potassium, magnesium)
    if pd.notnull(row.get('fiber')):
        score -= row['fiber'] * 1.2
    if pd.notnull(row.get('protein')):
        score -= row['protein'] * 0.8
    if pd.notnull(row.get('vitamin_c')):
        score -= row['vitamin_c'] * 0.05
    if pd.notnull(row.get('potassium')):
        score -= row['potassium'] * 0.001
    if pd.notnull(row.get('magnesium')):
        score -= row['magnesium'] * 0.02
    return score

def total_healthiness_score(row):
    ing_score = float(sum(score_ingredient(ing) for ing in row['ingredients_set']))
    nutri_score_val = float(score_nutrients(row))
    nutri_score_external = float(nutri_score(row))
    return ing_score + nutri_score_val - nutri_score_external

In [ ]:
# Filter foods with null or zero values for main 5 nutrients
main_fields = ['energy', 'protein', 'fiber', 'sugars', 'total_fat']
for field in main_fields:
    foods = foods[(foods[field].notnull()) & (foods[field] != 0)]

# Remove duplicate columns if any
foods = foods.loc[:, ~foods.columns.duplicated()]

# Debug: Check if foods DataFrame is empty after filtering
if foods.shape[0] == 0:
    print('No foods left after filtering. Check your filtering criteria or input data.')
else:
    print(f'Foods remaining after filtering: {foods.shape[0]}')
    print('Output type:', type(total_healthiness_score(foods.iloc[0])))
    print('Output value:', total_healthiness_score(foods.iloc[0]))

# Fix: Ensure total_healthiness_score returns a scalar float
def total_healthiness_score(row):
    ing_score = float(sum(score_ingredient(ing) for ing in row['ingredients_set']))
    nutri_score_val = float(score_nutrients(row))
    nutri_score_external = float(nutri_score(row))
    return float(ing_score + nutri_score_val - nutri_score_external)

# Calculate healthiness score for each food
if foods.shape[0] > 0:
    foods['healthiness_score'] = foods.apply(total_healthiness_score, axis=1)
else:
    print('No healthiness scores calculated because foods DataFrame is empty.')

Foods remaining after filtering: 1787
Output type: <class 'float'>
Output value: -43.23500000000001


In [98]:
# Display top and bottom foods by combined score
print('Top 10 Healthiest Foods (Combined Score):')
display(foods.sort_values('healthiness_score', ascending=False).head(40)[['description', 'healthiness_score', 'ingredients_set', 'protein', 'fiber', 'vitamin_c', 'potassium', 'magnesium', 'sugars', 'sodium', 'total_fat', 'cholesterol', 'saturated_fat']])

print('Top 10 Least Healthy Foods (Combined Score):')
display(foods.sort_values('healthiness_score').head(10)[['description', 'healthiness_score', 'ingredients_set', 'protein', 'fiber', 'vitamin_c', 'potassium', 'magnesium', 'sugars', 'sodium', 'total_fat', 'cholesterol', 'saturated_fat']])

Top 10 Healthiest Foods (Combined Score):


,description,healthiness_score,ingredients_set,protein,fiber,vitamin_c,potassium,magnesium,sugars,sodium,total_fat,cholesterol,saturated_fat
659,HEMP PROTEIN POWDER,114.3065,[hemp protein powder],38.71,35.5,NaN,1261.0,NaN,3.23,0.0,6.45,0.0,0.00
524,ESSENTIALS SHAKE DRINK MIX,106.7565,"[carrot powder, mushroom, broccoli, beet, oran...",55.56,11.1,66.7,889.0,222.0,2.78,611.0,11.11,0.0,1.39
3103,"LEMON SORBET WHEY PROTEIN POWDER BLEND, LEMON ...",104.7870,"[monk fruit extract, whey protein isolate blen...",80.65,3.2,0.0,NaN,NaN,3.23,129.0,3.23,65.0,0.00
619,"FRUIT AND GREENS SMOOTHIE POWDER, FRUIT AND GR...",101.9805,"[malic acid, thiamin hydrochloride, monk fruit...",32.89,16.4,631.6,559.0,NaN,29.61,757.0,3.29,0.0,0.00
3772,"CAFE LATTE 100% WHEY PROTEIN POWDER, CAFE LATTE",91.6750,"[whey protein isolate, xanthan gum, salt, sucr...",75.00,2.5,NaN,400.0,NaN,2.50,425.0,5.00,125.0,2.50
3801,CHOCOLATE HAZELNUT FLAVORED GOLD STANDARD 100%...,90.7265,"[lecithin, cocoa powder processed with alkali,...",72.73,3.0,NaN,727.0,NaN,3.03,485.0,4.55,106.0,1.52
4568,"LIGHT RED KIDNEY BEANS, LIGHT RED KIDNEY",87.0760,[light red kidney beans],22.86,31.4,NaN,886.0,NaN,2.86,14.0,1.43,0.0,0.00
3643,TOSTADAS,80.0705,"[stone ground corn, trace of calcium hydroxide...",16.67,44.4,NaN,67.0,NaN,11.11,111.0,27.78,0.0,5.56
2248,LIFT PROTEIN BAR,79.3555,"[caramel color, sunflower lecithin, tapioca st...",35.00,26.7,0.0,83.0,NaN,5.00,250.0,6.67,67.0,5.83
4186,"CHOCOLATE PROTEIN POWDER, CHOCOLATE",76.4480,"[protease, raw cane sugar, stevia leaf extract...",55.56,7.4,NaN,352.0,NaN,3.70,741.0,7.41,NaN,NaN


Top 10 Least Healthy Foods (Combined Score):


,description,healthiness_score,ingredients_set,protein,fiber,vitamin_c,potassium,magnesium,sugars,sodium,total_fat,cholesterol,saturated_fat
2923,"SCHLABACH AMISH BAKERY, ON-THE-GO POMEGRANATE ...",-1796.9320,"[cinnamon, all natural vanilla flavoring, appl...",9.09,9.1,0.0,NaN,NaN,23.64,300000.0,7.27,0.0,1.82
4187,"BELGIAN WAFFLE WITH SUGAR PEARLS, SUGAR PEARLS",-374.2365,"[lecithin, artificial flavor, eggs, yeast, sal...",5.56,1.1,NaN,111.0,NaN,23.33,189.0,24.44,33333.0,16.67
1890,FIG FRUIT PRESERVE,-234.3840,"[figs, pectin, lemon juice concentrate, sugar]",2.40,9.7,NaN,NaN,NaN,282.00,64.0,0.15,NaN,NaN
3632,"STRAWBERRY PROBIOTIC YOGURT DRINK, STRAWBERRY",-225.0540,"[sugar live, real strawberry cane, cultured gr...",100.00,10.0,NaN,NaN,NaN,316.67,717.0,81.67,267.0,48.33
1863,"GLAZED POPCORN, CLASSIC CARAMEL APPLE",-86.1690,"[vitamin ae, fructose, salt, sugar, fdc blue 1...",3.57,3.6,0.0,NaN,NaN,96.43,600.0,7.14,11.0,0.00
79,PALMER MERRY CHRISTMAS! DOUBLE CRISP,-85.6770,"[salt, and red #3, sugar, crisp rice rice, coc...",3.70,1.9,0.0,NaN,NaN,59.26,185.0,29.63,0.0,25.93
3313,ALMOND BERRY BAR,-83.3400,"[sunflower oil, soy lecithin added as emulsifi...",10.00,10.0,3.0,NaN,NaN,80.00,10.0,50.00,0.0,25.00
4253,"SNAP BAR, CREME DE MENTHE",-82.8985,"[milk protein concentrate, palm kernel and pal...",4.65,2.3,NaN,335.0,NaN,53.49,47.0,32.56,0.0,30.23
3367,DOUBLE CRISP CHOCOLATE N' SMOOTH BUNNIES,-82.3400,"[skim milk, barley malt, salt, sugar, crisp ri...",2.33,2.3,0.0,NaN,NaN,55.81,116.0,27.91,0.0,25.58
116,CREAMY MINTS IN PURE CHOCOLATE!,-78.1960,"[modified food starch, corn syrup, peppermint ...",2.63,2.6,0.0,NaN,NaN,81.58,66.0,7.89,0.0,6.58


In [ ]:
# Export scores 
output_dir = f'{path_prefix}/output/HealthinessScores'
if "hdfs" not in path_prefix:
    import os
    os.makedirs(output_dir, exist_ok=True)
foods[['fdc_id', 'description', 'healthiness_score']].to_csv(f'{output_dir}/healthiness_scores.csv', index=False)
print(f'Combined healthiness scores exported to {output_dir}/healthiness_scores.csv')

Combined healthiness scores exported to ../output/HealthinessScores/healthiness_scores.csv
